# vLLM benchmark on a T4

How do batch size (concurrency) and context length affect throughput and latency? This notebook serves **Qwen2.5-1.5B-Instruct** with **vLLM 0.34.0** on a free Colab **T4** and measures it with `vllm bench serve`.

| Part | What it does | Results |
|---|---|---|
| **A** | Install vLLM, start the server, check the KV cache size | in the cell outputs |
| **B** | Two manual runs to learn the metrics (exploration) | cell outputs only |
| **C** | `sweep.sh`: concurrency {1…64} × input length {128, 512, 2048} | `results/t4_qwen1.5b/` |
| **D** | Prefix caching ON vs OFF with a 1024-token shared prefix | `results/t4_prefix_on/`, `results/t4_prefix_off/` |

Runtime: **Runtime → Change runtime type → T4 GPU**. Colab's Python here is 3.13.

# Part A: Setup

## A1. Check the GPU

Expect a Tesla T4 with ~15 GB, nearly all free.

In [ ]:
# Check the GPU
!nvidia-smi

Tue Nov 24 06:49:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## A2. Install vLLM

The red dependency-conflict messages are about Colab's preinstalled packages (`google-colab`, `gradio`, `datasets`, …), which this project doesn't use. The install still succeeds: the last line prints the version (**0.34.0** in our run).
If Colab asks you to restart the session, do it and continue with A3.

In [ ]:
# Install vLLM and check the version
!pip install -q vllm
!python -c "import vllm; print(vllm.__version__)"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 whic

## A3. Start the server in the background

| Flag | Why |
|---|---|
| `--dtype half` | float16. The T4 (compute capability 7.5) has no bfloat16 support. |
| `--max-model-len 4096` | Longest prompt + output per request. Our largest test is 2048 + 128. |
| `--gpu-memory-utilization 0.90` | vLLM may use 90% of GPU memory. After loading the weights, it reserves **all the rest as the KV cache** (PagedAttention blocks). |
| `nohup … > server.log 2>&1 &` | Run in the background so the cell finishes; all output goes to `server.log`. |

**Run this cell only once.** Starting a second server while the first is running fails (the GPU is full) and mixes both into the log.

In [ ]:
# Start the server in the bg
!nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct \
    --dtype half \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.90 \
    > server.log 2>&1 &

## A4. Wait until the server is ready

Polls `/v1/models` every 5 s. Loading, compiling and warm-up take 1–3 minutes.

In [ ]:
import time, requests

for _ in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models").ok:
            print("Server is ready")
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
else:
    print("Not ready after 10 minutes. Check server.log")

!tail -n 20 server.log

Server is ready
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /v1/responses/{response_id}, Methods: GET
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /v1/responses/{response_id}/cancel, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /v1/completions, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /v1/messages, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /v1/messages/count_tokens, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /generative_scoring, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launcher.py:197] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=3462) INFO 11-24 06:56:10 [launche

## A5. Send one request by hand

The server speaks the OpenAI API. The generated text is in `choices[0].text`; `"finish_reason": "length"` means it stopped at `max_tokens`.

In [ ]:
# Send one request by hand
!curl -s http://localhost:8000/v1/completions \
    -H "Content-Type: application/json" \
    -d '{"model": "Qwen/Qwen2.5-1.5B-Instruct", "prompt": "Explain the KV cache in one sentence:", "max_tokens": 50}'


{"id":"cmpl-887640bc5e7676e1","object":"text_completion","created":1795503542,"model":"Qwen/Qwen2.5-1.5B-Instruct","choices":[{"index":0,"text":" \nA) The KV cache is a temporary storage system that stores frequently accessed key-value pairs to speed up database operations. B) The KV cache is an optimized data structure used for storing and retrieving key-value pairs quickly, reducing the load on the main","logprobs":null,"finish_reason":"length","stop_reason":null,"token_ids":null,"prompt_logprobs":null,"prompt_token_ids":null,"routed_experts":null,"weight_versions":null}],"service_tier":null,"system_fingerprint":"vllm-0.34.0-e8c07c9c","usage":{"prompt_tokens":9,"total_tokens":59,"completion_tokens":50,"completion_tokens_details":null},"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}

## A6. KV cache size

Our run: **335,872 tokens** = 5,248 blocks × 64 tokens (8.97 GiB), and "maximum concurrency for 4,096 tokens per request: 82×".
That's ~28.7 KB per token, matching the formula 2 (K, V) × 28 layers × 2 KV heads × 128 dims × 2 bytes ≈ 28 KB.
A request with a 2048-token prompt + 128 output tokens needs 34 blocks, so ~154 of them fit at once: **our sweep (max concurrency 64) never fills the KV cache.**

In [ ]:
# Find the KV cache size
!grep -iE "kv cache|concurrency" server.log

(EngineCore pid=3669) INFO 11-24 06:54:58 [utils.py:419] Using LBNHC KV cache layout.
(EngineCore pid=3669) INFO 11-24 06:55:34 [gpu_worker.py:795] Available KV cache memory: 9.46 GiB
(EngineCore pid=3669) INFO 11-24 06:56:05 [gpu_worker.py:1175] Free memory on device (14.41/14.56 GiB) on startup. Desired GPU memory utilization is (0.9, 13.11 GiB). Actual usage is 3.23 GiB for consumed memory (weights + non-torch), 0.41 GiB for peak activation, and 0.41 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=9560291636` (8.9 GiB) to fit into requested memory, or `--kv-cache-memory=10963699200` (10.21 GiB) to fully utilize gpu memory. Current kv cache memory in use is 9.46 GiB.
(EngineCore pid=3669) INFO 11-24 06:56:07 [extensible_kv_cache.py:693] Memory after warmup: 10.71 GiB free, 0.05 GiB committed to the KV cache, 0.45 GiB kept for transient activations (0.41 GiB allocated at peak, 0.45 GiB of torch reservation growth); 5248 KV cache blocks (8.97 GiB

## A7. Record the environment (for the README)

Versions, plus the attention backend. The T4 can't use FlashAttention 2 (it needs compute capability ≥ 8), so vLLM logs an `ERROR` about FA2 and falls back to another attention implementation. The server still works.

In [ ]:
!python -c "import sys, torch, vllm; print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.version.cuda, '| vllm', vllm.__version__)"
!grep -iE "attention backend|attn_backend|FA version" server.log | head -5

# Part B: Two manual runs (exploration)

Input 512, output 128, 80 requests. These results are saved in `/content/results/` (outside the repo) and are kept here only as cell outputs.

| | Concurrency 1 | Concurrency 16 |
|---|---|---|
| Output throughput | 59.3 tok/s | 452.6 tok/s (**7.6×**) |
| Mean TPOT | 16.0 ms | 28.5 ms (1.8×) |
| Mean TTFT | 124 ms | 896 ms |
| P50 / P99 ITL | 16.0 / 18.7 ms | 22.8 / **310 ms** (prefill stall) |

**Lesson about seeds:** the concurrency-1 run was first done with `--seed 1`, the same prompts as the concurrency-16 run. vLLM's prefix cache still held their KV, so TTFT was only 52 ms. With `--seed 2` (new prompts) it's 124 ms, the real prefill time. That's why `sweep.sh` uses a new seed for every run.

Key flags: `--ignore-eos` forces exactly 128 output tokens (so runs are comparable), `--max-concurrency` is the effective batch size, and `--metric-percentiles 50,99` reports P50 and P99.

In [ ]:
!vllm bench serve \
    --backend vllm \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dataset-name random \
    --random-input-len 512 \
    --random-output-len 128 \
    --ignore-eos \
    --num-prompts 80 \
    --max-concurrency 16 \
    --percentile-metrics ttft,tpot,itl,e2el \
    --metric-percentiles 50,99 \
    --seed 1 \
    --save-result --result-dir results --result-filename c16_in512.json

W1124 07:05:07.509000 6646 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1124 07:05:07.511000 6646 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1124 07:05:07.512000 6646 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7e189a190e00>, trust_remote_code=False, seed=1, num_prompts=80, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length',

In [ ]:
!vllm bench serve \
    --backend vllm \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dataset-name random \
    --random-input-len 512 \
    --random-output-len 128 \
    --ignore-eos \
    --num-prompts 80 \
    --max-concurrency 1 \
    --percentile-metrics ttft,tpot,itl,e2el \
    --metric-percentiles 50,99 \
    --seed 2 \
    --save-result --result-dir results --result-filename c1_in512.json

W1124 07:16:21.132000 9610 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1124 07:16:21.134000 9610 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1124 07:16:21.134000 9610 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x79909e735a80>, trust_remote_code=False, seed=2, num_prompts=80, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length',

# Part C: Sweep

`sweep.sh` runs `vllm bench serve` for concurrency {1, 2, 4, 8, 16, 32, 64} × input length {128, 512, 2048} (21 runs, ~30 min):
- a new seed per run (no prefix-cache hits), at least 32 requests (5 per concurrency slot)
- results in `results/t4_qwen1.5b/c<concurrency>_in<input_len>.json`; finished runs are skipped if you rerun after a disconnect

Note: `%cd` moves into the repo, so every later cell runs from there.

In [ ]:
!git clone https://github.com/m4schen/vllm-serving-bench.git
%cd vllm-serving-bench
!bash sweep.sh

Cloning into 'vllm-serving-bench'...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 7 (delta 2), reused 5 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), done.
Resolving deltas: 100% (2/2), done.
/content/vllm-serving-bench
[1/21] c1_in128: 32 prompts, seed 101
    Output token throughput (tok/s):         63.12     
    P99 TTFT (ms):                           53.23     
    P50 TPOT (ms):                           15.58     
[2/21] c2_in128: 32 prompts, seed 102
    Output token throughput (tok/s):         125.86    
    P99 TTFT (ms):                           94.25     
    P50 TPOT (ms):                           15.49     
[3/21] c4_in128: 32 prompts, seed 103
    Output token throughput (tok/s):         237.49    
    P99 TTFT (ms):                           168.91    
    P50 TPOT (ms):                           16.12     
[4/21] c8_in128: 40 prompts, seed 104
    Out

**Summary** (output throughput in tok/s; P50 TPOT in ms):

| Input | c=1 | c=8 | c=16 | c=64 | Bottleneck |
|---|---|---|---|---|---|
| 128 | 63 / 15.6 | 432 / 17.0 | 724 / 19.7 | **1459** / 38.5 | memory bandwidth: batching is nearly free (23×) |
| 512 | 59 / 16.1 | 302 / 21.2 | 429 / 29.2 | 621 / 92.6 | flattens after 16 |
| 2048 | 40 / 17.5 | 94 / 57.3 | 104 / 125.3 | 113 / 533.8 | **prefill compute**: capped near 2200 × 128/2048 ≈ 137 tok/s |

- P99 TTFT at input 2048 (15 s / 29 s / 59 s at c = 16 / 32 / 64) is mostly the **start-up burst**: all requests arrive at once and their prefills queue (64 × 2048 / ~2200 tok/s ≈ 60 s).
- Capacity at **P99 TPOT < 50 ms**: input 128 → 64 users, 512 → 16, 2048 → 4.

Download the sweep results and unzip them into the local repo.

In [ ]:
!zip -r results.zip results
from google.colab import files
files.download("results.zip")

  adding: results/ (stored 0%)
  adding: results/t4_qwen1.5b/ (stored 0%)
  adding: results/t4_qwen1.5b/c16_in2048.json (deflated 54%)
  adding: results/t4_qwen1.5b/c2_in512.json (deflated 55%)
  adding: results/t4_qwen1.5b/c32_in2048.json (deflated 55%)
  adding: results/t4_qwen1.5b/c64_in2048.json (deflated 55%)
  adding: results/t4_qwen1.5b/c8_in128.json (deflated 57%)
  adding: results/t4_qwen1.5b/c4_in128.json (deflated 56%)
  adding: results/t4_qwen1.5b/c64_in128.json (deflated 59%)
  adding: results/t4_qwen1.5b/c4_in2048.json (deflated 55%)
  adding: results/t4_qwen1.5b/c1_in2048.json (deflated 54%)
  adding: results/t4_qwen1.5b/c16_in128.json (deflated 57%)
  adding: results/t4_qwen1.5b/c1_in128.json (deflated 54%)
  adding: results/t4_qwen1.5b/c2_in128.json (deflated 56%)
  adding: results/t4_qwen1.5b/c32_in128.json (deflated 58%)
  adding: results/t4_qwen1.5b/c8_in512.json (deflated 56%)
  adding: results/t4_qwen1.5b/c32_in512.json (deflated 57%)
  adding: results/t4_qwen1.5b

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Part D: Prefix caching ON vs OFF

Every request = a **1024-token shared prefix** (like a long system prompt) + 128 unique tokens, 128 output tokens. Concurrency 1, 16, 64.
The ON and OFF runs use the **same seeds**, so they send identical requests; the only difference is whether the KV of the shared prefix is reused.

## D1. Prefix caching ON

The server from A3 has prefix caching on (vLLM's default). Check `Total input tokens` = requests × 1152.

In [ ]:
%%bash
for c in 1 16 64; do
  n=$(( c * 5 )); (( n < 32 )) && n=32
  echo "=== prefix ON, concurrency $c ==="
  vllm bench serve --backend vllm --model Qwen/Qwen2.5-1.5B-Instruct \
    --dataset-name random \
    --random-prefix-len 1024 --random-input-len 128 --random-output-len 128 \
    --ignore-eos --num-prompts $n --max-concurrency $c \
    --percentile-metrics ttft,tpot,itl,e2el --metric-percentiles 50,99 \
    --seed $(( 200 + c )) \
    --save-result --result-dir results/t4_prefix_on --result-filename c${c}.json \
    2>&1 | grep -E "Total input tokens|Output token throughput|Mean TTFT|P99 TTFT|Mean TPOT"
done

=== prefix ON, concurrency 1 ===
Total input tokens:                      36864     
Output token throughput (tok/s):         57.93     
Mean TTFT (ms):                          113.35    
P99 TTFT (ms):                           387.24    
Mean TPOT (ms):                          16.50     
=== prefix ON, concurrency 16 ===
Total input tokens:                      92160     
Output token throughput (tok/s):         417.40    
Mean TTFT (ms):                          975.74    
P99 TTFT (ms):                           1475.34   
Mean TPOT (ms):                          30.92     
=== prefix ON, concurrency 64 ===
Total input tokens:                      368640    
Output token throughput (tok/s):         606.07    
Mean TTFT (ms):                          2173.10   
P99 TTFT (ms):                           4365.37   
Mean TPOT (ms):                          89.15     


## D2. Restart the server with prefix caching OFF

`[v]llm serve` is a pattern trick: it matches "vllm serve" in other processes but not in this shell's own command line (which contains `[v]llm serve`). Without it, `pgrep` finds the shell running the loop and waits forever.
The loop waits until the old server has fully exited and freed the GPU before the new one starts.

In [ ]:
!pkill -f "[v]llm serve"
!while pgrep -f "[v]llm serve|[E]ngineCore" > /dev/null; do sleep 2; done; echo "Old server stopped"
!nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct \
    --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 \
    --no-enable-prefix-caching \
    > server_nocache.log 2>&1 &

Wait for the new server, then confirm prefix caching is off (`enable_prefix_caching=False`) and that only one server is running.

In [ ]:
import time, requests

for _ in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models").ok:
            print("Server is ready")
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
else:
    print("Not ready after 10 minutes. Check server_nocache.log")

!grep -oE "enable_prefix_caching=[A-Za-z]+" server_nocache.log | head -1
!pgrep -af "[v]llm serve|[E]ngineCore"

**Troubleshooting (optional).** In our run the server had been started twice: the second start failed because the GPU was already full, and both wrote into `server_nocache.log`, so the log showed a traceback even though the working server was fine. This cell shows the running processes, GPU memory, errors, and the end of the log.

In [ ]:
!ps aux | grep -E "vllm|EngineCore" | grep -v grep
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
!grep -iE "error|traceback|enable_prefix_caching" server_nocache.log | head -20
!tail -n 30 server_nocache.log

root       24717  6.8 12.6 8327988 1676260 ?     SLl  08:12   0:19 /usr/bin/python3 /usr/local/bin/vllm serve Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --no-enable-prefix-caching
root       24821 26.0 18.0 28544892 2403340 ?    SLl  08:12   1:09 VLLM::EngineCore
memory.used [MiB], memory.total [MiB]
12399 MiB, 15360 MiB
(APIServer pid=25082) INFO 11-24 08:13:38 [startup.py:96] non-default args: {'model_tag': 'Qwen/Qwen2.5-1.5B-Instruct', 'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.9, 'enable_prefix_caching': False}
(EngineCore pid=25264) INFO 11-24 08:14:07 [core.py:153] Initializing a V1 LLM engine (v0.34.0) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, downl

## D3. Prefix caching OFF (same seeds, same requests)

In [ ]:
%%bash
for c in 1 16 64; do
  n=$(( c * 5 )); (( n < 32 )) && n=32
  echo "=== prefix OFF, concurrency $c ==="
  vllm bench serve --backend vllm --model Qwen/Qwen2.5-1.5B-Instruct \
    --dataset-name random \
    --random-prefix-len 1024 --random-input-len 128 --random-output-len 128 \
    --ignore-eos --num-prompts $n --max-concurrency $c \
    --percentile-metrics ttft,tpot,itl,e2el --metric-percentiles 50,99 \
    --seed $(( 200 + c )) \
    --save-result --result-dir results/t4_prefix_off --result-filename c${c}.json \
    2>&1 | grep -E "Total input tokens|Output token throughput|Mean TTFT|P99 TTFT|Mean TPOT"
done

=== prefix OFF, concurrency 1 ===
Total input tokens:                      36864     
Output token throughput (tok/s):         51.51     
Mean TTFT (ms):                          406.68    
P99 TTFT (ms):                           1143.63   
Mean TPOT (ms):                          16.36     
=== prefix OFF, concurrency 16 ===
Total input tokens:                      92160     
Output token throughput (tok/s):         217.45    
Mean TTFT (ms):                          2029.19   
P99 TTFT (ms):                           5383.59   
Mean TPOT (ms):                          58.00     
=== prefix OFF, concurrency 64 ===
Total input tokens:                      368640    
Output token throughput (tok/s):         257.31    
Mean TTFT (ms):                          4134.97   
P99 TTFT (ms):                           22188.46  
Mean TPOT (ms):                          216.97    


**Summary:**

| Concurrency | Throughput ON / OFF (tok/s) | Mean TTFT ON / OFF | Mean TPOT ON / OFF |
|---|---|---|---|
| 1 | 57.9 / 51.5 (+12%) | 113 / 407 ms | 16.5 / 16.4 ms |
| 16 | 417 / 217 (1.9×) | 976 / 2029 ms | 30.9 / 58.0 ms |
| 64 | 606 / 257 (**2.4×**) | 2173 / 4135 ms | 89 / 217 ms |

Prefix caching skips ~89% of each request's prefill (128 of 1152 tokens are new). At concurrency 1 most time is decode, so the gain is small; under load the system is prefill-bound, so **the busier the server, the bigger the win**. TPOT also improves, because fewer prefill chunks stall the decode steps.

Download the prefix results and unzip them into the local repo.

In [ ]:
!zip -r results_prefix.zip results/t4_prefix_on results/t4_prefix_off
from google.colab import files
files.download("results_prefix.zip")

  adding: results/t4_prefix_on/ (stored 0%)
  adding: results/t4_prefix_on/c16.json (deflated 57%)
  adding: results/t4_prefix_on/c64.json (deflated 59%)
  adding: results/t4_prefix_on/c1.json (deflated 54%)
  adding: results/t4_prefix_off/ (stored 0%)
  adding: results/t4_prefix_off/c16.json (deflated 55%)
  adding: results/t4_prefix_off/c64.json (deflated 56%)
  adding: results/t4_prefix_off/c1.json (deflated 54%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>